# Student Bags: Price Analysis
## Problem Statement
Students frequently need to carry a laptop and deal with rain at the start of 2026. They have to prioritize bags equipped with a laptop compartment and a waterproof feature.

Are these two needs (Waterproof & Laptop Compartment) reflected in the bag's price? Or are there other attributes that matter more?

The dataset consists of 12,805 rows & 10 columns.

*This notebook merges Mini Project 1 (Data Cleaning) and Mini Project 2 (Data Visualization & EDA) into a single, continuous workflow.*

# DATA UNDERSTANDING

## IMPORT & LOAD

In [1]:
import pandas as pd # import and analyze tabular data
import numpy as np # for computing the PMF (statistical analysis)
import matplotlib.pyplot as plt # for basic visualizations
import seaborn as sns # for statistical visualizations
import plotly.express as px # interactive visualization
import plotly.io as pio # Plotly renderer setting
pio.renderers.default = 'iframe'

df=pd.read_csv('tsb.csv') # load the original dataset already stored in Jupyter. File renamed to "tsb" for convenience.
df_asli=df.copy() # create a backup dataset to be used later as a before/after comparison after preprocessing.

df.head() # show the first 5 rows. Useful for seeing what the dataset structure looks like.

,Brand,Material,Size,Compartments,Laptop Compartment,Waterproof,Style,Color,Weight Capacity (kg),Price
0,Nike,Nylon,Small,9.0,No,No,Tote,Black,9.333159,116.655111
1,Adidas,Polyester,Large,NaN,No,No,Backpack,Gray,26.258602,127.225657
2,Under Armour,Nylon,Medium,7.0,No,No,Backpack,Blue,15.049938,132.131249
3,Under Armour,Canvas,Large,8.0,Yes,No,Tote,Green,17.426660,NaN
4,Puma,NaN,Large,1.0,No,Yes,Backpack,Black,12.771516,86.334448


## INITIAL INFORMATION

In [2]:
df.info() # show basic dataset information. Main point: number of rows, columns, data types, and non-null count per column

# All 10 columns have missing values (Non-Null < 12,805)
# Compartments and Weight Capacity are float64, but should be integer (a bag can't have half a compartment)
# All categorical columns are object type and will be converted to category for memory efficiency
# Price is the main variable to be analyzed

<class 'pandas.DataFrame'>
RangeIndex: 12805 entries, 0 to 12804
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Brand                 12152 non-null  str    
 1   Material              12167 non-null  str    
 2   Size                  12169 non-null  str    
 3   Compartments          12161 non-null  float64
 4   Laptop Compartment    12156 non-null  str    
 5   Waterproof            12140 non-null  str    
 6   Style                 12133 non-null  str    
 7   Color                 12144 non-null  str    
 8   Weight Capacity (kg)  12154 non-null  float64
 9   Price                 12181 non-null  float64
dtypes: float64(3), str(7)
memory usage: 1000.5 KB


In [3]:
df.describe() # show basic statistical information. Key point: see the range and distribution of numeric columns

,Compartments,Weight Capacity (kg),Price
count,12161.000000,12154.000000,12181.000000
mean,5.536058,17.523785,81.748538
std,2.859863,7.559455,39.271530
min,1.000000,5.000000,15.000000
25%,3.000000,11.030286,47.379405
50%,6.000000,17.591189,81.312176
75%,8.000000,24.167772,115.715245
max,10.000000,30.000000,150.000000


## CHECKING MISSING VALUES

In [4]:
print(df.isnull().sum()) # more detailed check of missing values in each column

# each column with missing values represents ~5% of total data
# combined, this represents ~40% of total data
# missing values need to be handled

Brand                   653
Material                638
Size                    636
Compartments            644
Laptop Compartment      649
Waterproof              665
Style                   672
Color                   661
Weight Capacity (kg)    651
Price                   624
dtype: int64


## CHECKING FOR TYPOS

In [5]:
print(df['Brand'].value_counts()) # checking for typos in the Brand variable

# we can see there are no typos (in spelling or pronunciation)

Brand
Under Armour    2472
Puma            2434
Nike            2427
Jansport        2420
Adidas          2399
Name: count, dtype: int64


In [6]:
print(df['Material'].value_counts()) # checking for typos in the Material variable

# no typos found

Material
Polyester    3099
Leather      3050
Canvas       3018
Nylon        3000
Name: count, dtype: int64


In [7]:
print(df['Size'].value_counts()) # checking for typos in the Size variable

# no typos found

Size
Large     4083
Small     4046
Medium    4040
Name: count, dtype: int64


In [8]:
print(df['Style'].value_counts()) # checking for typos in the Style variable

# no typos found

Style
Tote         4073
Messenger    4050
Backpack     4010
Name: count, dtype: int64


In [9]:
print(df['Laptop Compartment'].value_counts()) # checking for typos in the Laptop Compartment variable

# no typos found

Laptop Compartment
No     6140
Yes    6016
Name: count, dtype: int64


In [10]:
print(df['Waterproof'].value_counts()) # checking for typos in the Waterproof variable

# no typos found

Waterproof
No     6167
Yes    5973
Name: count, dtype: int64


In [11]:
print(df['Color'].value_counts()) # checking for typos in the Color variable

# no typos found

Color
Gray     2077
Green    2061
Black    2049
Blue     2003
Pink     1982
Red      1972
Name: count, dtype: int64


## CHECKING FOR DUPLICATES

In [12]:
duplicate=df.duplicated().sum() # checking for duplicates in the dataset
print(duplicate)

# no duplicate data found in the dataset

0


# DATA PREPARATION

## HANDLING MISSING VALUES

In [13]:
df['Brand']=df['Brand'].fillna('Unknown') # fill with "Unknown" for 2 variables: Brand and Color.
df['Color']=df['Color'].fillna('Unknown')

for col in ['Material', 'Size', 'Style',
            'Laptop Compartment', 'Waterproof']:
    df[col]=df[col].fillna(df[col].mode()[0]) 

# fill the remaining 5 variables with the mode.
# these variables have few, well-defined categories with a near-uniform distribution. Every bag must be made of something, must have a size, and must have a style.
# want to preserve the Yes/No structure for 0/1 encoding (Waterproof & Laptop Compartment variables)

df['Compartments']=df['Compartments'].fillna(df['Compartments'].mode()[0])

# also fill with the mode because Compartments is a whole number. Filling with the mean could introduce decimal values.

df['Weight Capacity (kg)']=df['Weight Capacity (kg)'].fillna(
    df['Weight Capacity (kg)'].median()
)

# fill Weight Capacity with the median because it's a continuous variable where the median is not affected by extreme values.

# leave Price missing values empty because Price is the key variable in this analysis, and filling it with the mean/median could distort the data's distribution.

print(df.isnull().sum()) # confirm that the missing value filling process is complete. Now only Price still has missing values.

Brand                     0
Material                  0
Size                      0
Compartments              0
Laptop Compartment        0
Waterproof                0
Style                     0
Color                     0
Weight Capacity (kg)      0
Price                   624
dtype: int64


## STANDARDIZING FORMAT & DATA TYPES

In [14]:
df.columns=df.columns.str.strip() # remove extra whitespace
df.columns=df.columns.str.replace(' ', '_').str.replace(r'[()]', '', regex=True)
# standardize column names: remove spaces and parentheses for easier coding

df['Weight_Capacity_kg']=df['Weight_Capacity_kg'].round(2) # round to 2 decimal places, sufficient precision for weight in kg

for col in ['Brand', 'Material', 'Size', 'Style', 'Color',
            'Laptop_Compartment', 'Waterproof']:
    df[col]=df[col].astype('category')

# convert categorical columns to the category dtype for memory efficiency

In [15]:
df['Compartments']=df['Compartments'].astype(int) # convert to integer since a bag cannot have a fractional number of compartments

df.info() # check the conversion result

<class 'pandas.DataFrame'>
RangeIndex: 12805 entries, 0 to 12804
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   Brand               12805 non-null  category
 1   Material            12805 non-null  category
 2   Size                12805 non-null  category
 3   Compartments        12805 non-null  int64   
 4   Laptop_Compartment  12805 non-null  category
 5   Waterproof          12805 non-null  category
 6   Style               12805 non-null  category
 7   Color               12805 non-null  category
 8   Weight_Capacity_kg  12805 non-null  float64 
 9   Price               12181 non-null  float64 
dtypes: category(7), float64(2), int64(1)
memory usage: 388.1 KB


## SUMMARY

In [16]:
print("Dataset Condition (Post-Cleaning)")
print(f"Shape               : {df.shape}")
print(f"Missing Values      : {df.isnull().sum().sum()} (Price, 624 rows)")
print(f"Duplicates          : {df.duplicated().sum()}")

# Price NaN is intentionally left unfilled to preserve the distribution and avoid disrupting the visualization stage
# if all NaN rows were dropped from the raw data, ~40% of the data would be lost.
# decision: fill per column, then drop Price missing values (new dataframe called df_viz) for visualization and analysis.

# key variables:
# Price            : the variable being analyzed
# functional features : Waterproof, Laptop_Compartment, nilai_fungsional
# physical features   : Size, Weight_Capacity_kg, Compartments
# identity features   : Brand, Material, Style, Color

display(df.head())
display(df.describe())

Dataset Condition (Post-Cleaning)
Shape               : (12805, 10)
Missing Values      : 624 (Price, 624 rows)
Duplicates          : 0


,Brand,Material,Size,Compartments,Laptop_Compartment,Waterproof,Style,Color,Weight_Capacity_kg,Price
0,Nike,Nylon,Small,9,No,No,Tote,Black,9.33,116.655111
1,Adidas,Polyester,Large,5,No,No,Backpack,Gray,26.26,127.225657
2,Under Armour,Nylon,Medium,7,No,No,Backpack,Blue,15.05,132.131249
3,Under Armour,Canvas,Large,8,Yes,No,Tote,Green,17.43,NaN
4,Puma,Polyester,Large,1,No,Yes,Backpack,Black,12.77,86.334448


,Compartments,Weight_Capacity_kg,Price
count,12805.000000,12805.000000,12181.000000
mean,5.509098,17.527142,81.748538
std,2.789476,7.364781,39.271530
min,1.000000,5.000000,15.000000
25%,3.000000,11.420000,47.379405
50%,5.000000,17.590000,81.312176
75%,8.000000,23.800000,115.715245
max,10.000000,30.000000,150.000000
